In [1]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import optuna
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input
from tensorflow.keras.layers import ConvLSTM1D, Flatten, Dense

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

class WeightLogger(tf.keras.callbacks.Callback):
    def __init__(self, layer_index=0):
        self.layer_index = layer_index
        self.weights_per_epoch = []

    def on_epoch_end(self, epoch, logs=None):
        weights = self.model.layers[self.layer_index].get_weights()[0]
        self.weights_per_epoch.append(weights.copy())


def load_and_prepare_data(csv_path, production_column='production', window_size=42):
    df = pd.read_csv(csv_path, sep=',')
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(df.values)
    target_scaler = MinMaxScaler()
    target_scaler.fit(df[[production_column]])
    target_col_idx = df.columns.get_loc(production_column)
    x, y = [], []
    for i in range(window_size, len(data_scaled)):
        x.append(data_scaled[i-window_size:i])
        y.append(data_scaled[i, target_col_idx])
    x, y = np.array(x), np.array(y)
    train_split_index = int(0.8 * len(x))
    test_split_index = int(0.9 * len(x))
    x_train, y_train = x[:train_split_index], y[:train_split_index]
    x_test, y_test = x[train_split_index:test_split_index], y[train_split_index:test_split_index]
    x_val, y_val = x[test_split_index:], y[test_split_index:]
    x_train_conv = np.expand_dims(x_train, axis=2)
    x_test_conv = np.expand_dims(x_test, axis=2)
    x_val_conv = np.expand_dims(x_val, axis=2)
    return x_train_conv, y_train, x_test_conv, y_test, x_val_conv, y_val, df, target_scaler


def build_convlstm_model(lr, filters1, filters2, dense_units, input_shape):
    model = Sequential([
        ConvLSTM1D(filters=int(filters1), kernel_size=(1,), activation='tanh',
                   return_sequences=True, input_shape=input_shape),
        ConvLSTM1D(filters=int(filters2), kernel_size=(1,), activation='tanh', return_sequences=False),
        Flatten(),
        Dense(units=int(dense_units), activation='relu'),
        Dense(1, activation="linear")
    ])
    optimizer = Adam(learning_rate=lr)
    model.compile(loss="mae", optimizer=optimizer)
    return model


def train_and_evaluate_model(model, x_train, y_train, x_val, y_val,
                             epochs=20, batch_size=512, verbose=0, dataset_name="dataset"):
    stop_early = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
    weight_logger = WeightLogger(layer_index=0)
    start_time = time.time()
    history = model.fit(x_train, y_train,
                        validation_data=(x_val, y_val),
                        epochs=epochs,
                        batch_size=batch_size,
                        verbose=verbose,
                        callbacks=[stop_early, weight_logger])
    training_time = time.time() - start_time
    w00 = [w[0, 0] for w in weight_logger.weights_per_epoch]
    plt.figure(figsize=(8, 4))
    plt.plot(w00)
    plt.xlabel("Époque")
    plt.ylabel("Poids [0,0]")
    plt.title("Évolution du poids [0,0]")
    plt.grid(True)
    plt.savefig(f"{dataset_name}_poids_w00e350_ws42.png")
    plt.close()
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Validation')
    plt.title("Loss par époque")
    plt.xlabel("Époque")
    plt.ylabel("MAE")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{dataset_name}_loss_curvee350_ws42.png")
    plt.close()
    return history, training_time, weight_logger


def inference_and_plot(model, x_test, y_test, target_scaler, dataset_name): 
    preds = model.predict(x_test)
    y_test_real = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_real = target_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test_real, y_pred_real)
    mse = mean_squared_error(y_test_real, y_pred_real)
    r2 = r2_score(y_test_real, y_pred_real)
    plt.figure(figsize=(10, 6))
    plt.plot(y_test_real, label="Vrai")
    plt.plot(y_pred_real, label="Prévu")
    plt.legend()
    plt.title(f"{dataset_name} - MAE: {mae:.4f} | R²: {r2:.4f} | MSE: {mse:.4f}")
    plt.grid(True)
    plt.savefig(f"{dataset_name}_courbe_perfe350_ws42.png")
    plt.close()
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test_real, y_pred_real, alpha=0.7, color='orange')
    plt.plot([min(y_test_real), max(y_test_real)], [min(y_test_real), max(y_test_real)], 'r--')
    plt.xlabel("Réel")
    plt.ylabel("Prédit")
    plt.title("Scatter plot")
    plt.grid(True)
    plt.savefig(f"{dataset_name}_scatter_perfe350_ws42.png")
    plt.close()
    errors = y_test_real - y_pred_real
    plt.figure(figsize=(8, 4))
    plt.hist(errors, bins=30, color='orange', edgecolor='black')
    plt.title("Histogramme des erreurs")
    plt.grid(True)
    plt.savefig(f"{dataset_name}_hist_errorse350_ws42.png")
    plt.close()
    # Corriger division par zéro pour les erreurs en pourcentage
    safe_y_test_real = np.where(y_test_real == 0, np.nan, y_test_real)
    
    df_stats = pd.DataFrame({
        "Y_test": y_test_real,
        "Y_pred": y_pred_real,
        "Error": errors,
        "Error_Percent": np.abs(errors) / safe_y_test_real * 100
    })
    
    # Sauvegarder les statistiques sans générer d'avertissements
    df_stats.describe().to_csv(f"{dataset_name}_stats_erreurse350_ws42.csv")
    return mae, mse, r2


def objective(trial, input_shape, x_train, y_train, x_val, y_val):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    f1 = trial.suggest_int('filters1', 32, 128)
    f2 = trial.suggest_int('filters2', 32, 128)
    dense = trial.suggest_int('dense_units', 32, 128)
    model = build_convlstm_model(lr, f1, f2, dense, input_shape)
    history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                        epochs=150, batch_size=256, verbose=0)
    return min(history.history['val_loss'])


def run_experiment(csv_path, dataset_name):
    x_train, y_train, x_test, y_test, x_val, y_val, df, target_scaler = load_and_prepare_data(csv_path)
    input_shape = x_train.shape[1:]
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective(trial, input_shape, x_train, y_train, x_val, y_val), n_trials=15)

    best = study.best_params
    print("Meilleurs hyperparamètres trouvés :")
    for k, v in best.items():
        print(f"  {k} = {v}")
        
    # Sauvegarde dans un fichier .txt
    with open(f"{dataset_name}_best_params_e350_ws42.txt", "w") as f:
        f.write("Meilleurs hyperparamètres trouvés :\n")
        for k, v in best.items():
            f.write(f"{k} = {v}\n")

    model = build_convlstm_model(best['lr'], best['filters1'], best['filters2'], best['dense_units'], input_shape)
    history, training_time, _ = train_and_evaluate_model(model, x_train, y_train, x_val, y_val,
                                                         epochs=350, dataset_name=dataset_name)
    mae, mse, r2 = inference_and_plot(model, x_test, y_test, target_scaler, dataset_name)
    print(f"{dataset_name} — MAE: {mae:.4f}, MSE: {mse:.4f}, R²: {r2:.4f}")

if __name__ == "__main__":
    run_experiment("../DataCleaning/scaled_dataset.csv", "scaled_dataset")

2025-06-25 09:39:24.368911: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-25 09:39:24.432115: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-25 09:39:26.159225: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/sismail/py_envs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning:

Meilleurs hyperparamètres trouvés :
  lr = 0.007954235411633086
  filters1 = 67
  filters2 = 112
  dense_units = 117
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step
scaled_dataset — MAE: 1.8017, MSE: 12.4576, R²: 0.9614
